In [2]:
import os, sys, importlib
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(".."))
from utils import modeling

## test:

In [3]:
path_listings_filtered="mod_results\listings_tactics_filtered.csv"
df_raw=pd.read_csv(path_listings_filtered)
df=df_raw.copy()
print(len(df))
print(df.columns)
display(df.head())

37013
Index(['Unnamed: 0', 'id', 'listing_url', 'scrape_id', 'last_scraped',
       'source', 'name', 'description', 'neighborhood_overview', 'picture_url',
       ...
       'auto_promotion_weighted', 'exemplarité_weighted', 'has_rating',
       'years_since_host', 'has_since', 'lang', 'len', 'has_response_rate',
       'professional_host', 'booking_rate_l30d'],
      dtype='object', length=117)


,Unnamed: 0,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,...,auto_promotion_weighted,exemplarité_weighted,has_rating,years_since_host,has_since,lang,len,has_response_rate,professional_host,booking_rate_l30d
0,0,3109.0,https://www.airbnb.com/rooms/3109,2.023121e+13,2023-12-12,city scrape,Rental unit in Paris · ★5.0 · 1 bedroom · 1 be...,NaN,Good restaurants<br />very close the Montparna...,https://a0.muscache.com/pictures/baeae9e2-cd53...,...,NaN,NaN,1,16.0,1,no_text,0,1,0,0.000000
1,2,81106.0,https://www.airbnb.com/rooms/81106,2.023121e+13,2023-12-13,city scrape,Rental unit in Paris · ★4.84 · 1 bedroom · 1 b...,NaN,The neighborhood will show you an other side o...,https://a0.muscache.com/pictures/miso/Hosting-...,...,0.606779,-0.441423,1,13.0,1,en,47,1,0,0.000000
2,3,7397.0,https://www.airbnb.com/rooms/7397,2.023121e+13,2023-12-13,city scrape,Rental unit in Paris · ★4.73 · 2 bedrooms · 2 ...,NaN,NaN,https://a0.muscache.com/pictures/67928287/330b...,...,-1.456089,-0.778443,1,16.0,1,en,10,1,1,0.333333
3,5,81615.0,https://www.airbnb.com/rooms/81615,2.023121e+13,2023-12-13,city scrape,Rental unit in Paris · ★4.74 · 1 bedroom · 1 b...,NaN,NaN,https://a0.muscache.com/pictures/2630947/9340f...,...,-0.546908,0.839862,1,13.0,1,en,101,1,0,0.000000
4,9,86053.0,https://www.airbnb.com/rooms/86053,2.023121e+13,2023-12-13,city scrape,Rental unit in Paris · ★4.88 · 1 bedroom · 2 b...,NaN,The flat is situated in the very heart of Pari...,https://a0.muscache.com/pictures/1982057/f94cf...,...,1.246919,0.084940,1,14.0,1,en,159,1,0,0.000000


In [4]:
# all_host_vars= ['host_picture_url', 'host_identity_verified', 'number_of_reviews', 'host_is_superhost',
#            'review_scores_rating', 'has_rating', 'host_since', 'years_since_host', 'host_about', 
#            'has_host_about', 'lang', 'len', 'host_response_time', 'host_response_rate', 'has_response_rate',
#            'calculated_host_listings_count', 'professional_host']
group_col='host_is_superhost'

host_vars= ['number_of_reviews', 'review_scores_rating', 'has_rating', 
        'host_since', 'years_since_host', 'host_about', 'has_host_about', 'lang', 'len',
        'host_response_time', 'host_response_rate', 'has_response_rate',
        'professional_host']

proxy_obj_vars=['price', "availability_30", "number_of_reviews_l30d","booking_rate_l30d",
        "room_type", "minimum_nights","instant_bookable"]

cols_tactics=['ouverture_mean',"authenticité_mean","sociabilité_mean", "auto_promotion_mean","exemplarité_mean",
        "ouverture_weighted", "authenticité_weighted","sociabilité_weighted","auto_promotion_weighted","exemplarité_weighted",
        ]


In [ ]:
##VIF: vérifier la corélation entre les variables:
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices

# formula=('booking_rate_l30d ~ C(langue) + C(host_is_superhost) + C(professional_host) +'
# ' yrs_experience + C(host_identity_verified)+ C(host_has_profile_pic) +'
# ' availability_30 + price + review_scores_rating + '
# 'C(host_location_category) + C(property_type_category) + C(instant_bookable) + '
# 'len + openness +authenticity + sociability + self_promotion + exemplification ')


# formula=('booking_rate_l30d ~'
#         'C(host_identity_verified)+ C(host_has_profile_pic) +' 
#         'number_of_reviews + review_scores_rating + C(has_rating) +' 
#         ' years_since_host + C(has_host_about)+ C(lang) + len +'
#         'C (host_response_time) + host_response_rate + C(has_response_rate) + C(professional_host) +'
#         'price + booking_rate_l30d + C(room_type) + minimum_nights + C(instant_bookable)+'
#         'ouverture_mean + authenticité_mean + sociabilité_mean + auto_promotion_mean + exemplarité_mean'
# )


# has_host_about在lang:no_text中重复了
# review_scores_rating的0 和has_raitng重复

formula=('booking_rate_l30d ~'
        'number_of_reviews + review_scores_rating +' 
        ' years_since_host + C(lang) + len + C(professional_host) + C(host_is_superhost)+'
        'price + C(room_type) + minimum_nights + C(instant_bookable)+'
        'ouverture_mean + authenticité_mean + sociabilité_mean + auto_promotion_mean + exemplarité_mean'
)


y, X = dmatrices(formula, data=df, return_type='dataframe')
vif_df = pd.DataFrame()
vif_df['Variables']=X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_df['niveau_colinearite'] = vif_df["VIF"].apply(
    lambda x: " " if x <= 1 else
              "*" if x <= 5 else
              "**" if x <= 10 else
              "***"
)
display(vif_df)  


d:\miniconda3\envs\airbnb_env\lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
d:\miniconda3\envs\airbnb_env\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,Variables,VIF,niveau_colinearite
0,Intercept,1.608709e+04,***
1,C(lang)[T.fr],1.252065e+00,*
2,C(lang)[T.no_text],NaN,***
3,C(lang)[T.other_langs],1.024817e+00,*
4,C(professional_host)[T.1],1.381972e+00,*
5,C(host_is_superhost)[T.f],inf,***
6,C(host_is_superhost)[T.t],inf,***
7,C(room_type)[T.Entire home/apt],inf,***
8,C(room_type)[T.Hotel room],inf,***
9,C(room_type)[T.Private room],inf,***


In [59]:
df.host_is_superhost.value_counts(dropna=False)

host_is_superhost
f                     29280
t                      7733
Entire rental unit        1
Name: count, dtype: int64

In [ ]:
# model1=smf.ols('booking_rate_l30d ~ C(langue) + C(host_is_superhost) + C(professional_host) +'
# ' yrs_experience + C(host_identity_verified)+ C(host_has_profile_pic) +'#host_response_rate+
# ' availability_30 + price + review_scores_rating + '
# 'C(host_location_category) + C(property_type_category)+ C(instant_bookable) + '#+ beds +bathrooms+
# 'len + len_squared', data=df).fit()

# 'booking_rate_l30d ~'
# 'number_of_reviews + review_scores_rating +' 
# ' years_since_host + C(lang) + len + C(professional_host) +'
# 'price + C(room_type) + minimum_nights + C(instant_bookable)+'

        
model_basic=smf.ols('booking_rate_l30d ~'
        'number_of_reviews + review_scores_rating + C(has_rating) +' 
        ' years_since_host +'
        'C(professional_host) +'
        'price + C(room_type) + minimum_nights + C(instant_bookable)'
        , data=df).fit()

print(model_basic.summary())

                            OLS Regression Results                            
Dep. Variable:      booking_rate_l30d   R-squared:                       0.144
Model:                            OLS   Adj. R-squared:                  0.143
Method:                 Least Squares   F-statistic:                     259.3
Date:                Wed, 10 Dec 2025   Prob (F-statistic):               0.00
Time:                        17:41:36   Log-Likelihood:                 8549.2
No. Observations:               37012   AIC:                        -1.705e+04
Df Residuals:                   36987   BIC:                        -1.684e+04
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [ ]:
formula='booking_rate_l30d ~'
        'C(host_identity_verified)+ C(host_has_profile_pic) +' 
        'number_of_reviews + review_scores_rating + C(has_rating) +' 
        ' years_since_host + C(has_host_about)+ C(lang) + len +'
        'C (host_response_time) + host_response_rate + C(has_response_rate) + C(professional_host) +'
        'price + C(room_type) + minimum_nights + C(instant_bookable)+'
# "ouverture_mean + authenticité_mean + sociabilité_mean + auto_promotion_mean + exemplarité_mean"

vars_tactics=['ouverture_mean',"authenticité_mean","sociabilité_mean", "auto_promotion_mean","exemplarité_mean"]
formula+='+'.join(vars_tactics)
print(f"[INFO] formula : {formula}")


model_tactics_mean=smf.ols(formula, data=df).fit()
print(model_tactics_mean.summary())

IndentationError: unexpected indent (2292026982.py, line 2)

In [ ]:
'ouverture_mean',"authenticité_mean","sociabilité_mean", "auto_promotion_mean","exemplarité_mean",

In [ ]:
# model_tactics_weighted=smf.ols('booking_rate_l30d ~'
#         'C(host_identity_verified)+ C(host_has_profile_pic) +' 
#         'number_of_reviews + review_scores_rating + C(has_rating) +' 
#         ' years_since_host + C(has_host_about)+ C(lang) + len +'
#         'C (host_response_time) + host_response_rate + C(has_response_rate) + C(professional_host) +'
#         'price + C(room_type) + minimum_nights + C(instant_bookable)+'
#         "ouverture_weighted + authenticité_weighted + sociabilité_weighted + auto_promotion_weighted + exemplarité_weighted"
#         , data=df).fit()
# print(model_tactics_weighted.summary())


                            OLS Regression Results                            
Dep. Variable:      booking_rate_l30d   R-squared:                       0.169
Model:                            OLS   Adj. R-squared:                  0.167
Method:                 Least Squares   F-statistic:                     120.5
Date:                Wed, 10 Dec 2025   Prob (F-statistic):               0.00
Time:                        17:46:14   Log-Likelihood:                 3609.6
No. Observations:               16081   AIC:                            -7163.
Df Residuals:                   16053   BIC:                            -6948.
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------